In [1]:
import pandas as pd
from thefuzz import process
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from scipy import stats
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from typing import Dict

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor


In [2]:
train_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/train.csv")

In [3]:
train_df.set_index("carID")

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,C Class,2015.0,13498,Manual,14480.0,etrol,125.0,53.300000,2.0,78.0,0.000000,0.0
6265,Audi,Q3,2013.0,12495,Semi-Auto,52134.0,Diesel,200.0,47.900000,2.0,38.0,2.000000,0.0
54886,Toyota,Aygo,2017.0,8399,Automatic,11304.0,Petrol,145.0,67.000000,1.0,57.0,3.000000,0.0


# Data cleaning

In [ ]:
# TODO: Move the functions to a separate file

In [4]:
# String cleaning and Small numbers changes

def simple_processing(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don"t require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    # Get the most frequent brand for each model -> returns df with model and brand
    brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model", how="left", suffixes=("", "_mode"))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand"] = df["Brand"].fillna(df["Brand_mode"])  # rename new column
    df.drop("Brand_mode", axis=1, inplace=True)  # remove the old brand column
    
    ################################################################################
    # Simple Number Cleaning
    ################################################################################

    # Cleaning numeric columns
    df["year"] = df["year"].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)

    # Round year to integer (no fractional years)
    df["year"] = df["year"].round()
    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df["mileage"] = abs(df["mileage"].round())
    
    # Tax: take absolute value and round
    df["tax"] = abs(df["tax"].round())
    
    # MPG: round to 1 decimal place 
    df["mpg"] = abs(df["mpg"].round(1))
    
    # Engine size: round to 1 decimal place
    df["engineSize"] = abs(df["engineSize"].round(1))
    
    # Paint quality correction (domain-specific business logic)
    # Assumption based on data exploration:
    # - Values < 4 likely had decimal point in wrong place (e.g., 3.5 -> 35%)
    # - Values > 100 likely have erroneous leading 1 (e.g., 185 -> 85%)
    def fix_paint_quality(x):
        if x < 4:
            return x * 10
        elif x > 100:
            return x - 100
        else:
            return x
    
    df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
    
    # Previous owners: take absolute value and round to integer
    df["previousOwners"] = abs(df["previousOwners"].round())
    
    return df

In [5]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [6]:
# Train imputer on train

def fit_imputer(df, fast=True): 
    # Select estimator based on speed/accuracy tradeoff
    if fast:
        # Use default BayesianRidge (fast, ~1 second)
        estimator = None
    else:
        # Use Random Forest for better accuracy with complex relationships (~2 minutes)
        estimator = RandomForestRegressor(
            n_estimators=20,      # Limited trees for speed
            max_depth=10,         # Prevent overfitting
            random_state=12       # Reproducibility
        )

    # TODO: for later
    # Test if we perform better if we use numerical and categorical imputers separately
    
    # Initialize imputer
    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="mean"         # Initial fill before iterative process
    )
    
    # TODO: Test other categorical imputers like: missForest, datawig
    """imputer = IterativeImputer(
        estimator=RandomForestClassifier(),
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="most_frequent"         # Initial fill before iterative process
        )"""

    # FIT on training data
    # CRITICAL: We fit on data that still has missing values!
    # The imputer learns patterns of missingness and relationships

    imputer.fit(df)
    
    return imputer

In [7]:
def apply_imputer(df, imputer):
    """
        Apply the pretrained imputer to the dataframe
    """

    imputed_values = imputer.transform(df)

    df[df.columns] = imputed_values

    # TODO: rounding is not the best approach as the imputers prediction are continues thus 1.2 doesnt mean the value is closer to 1 than 2
    # However, rounding is the quickest way to fix this for now
    df[["mpg", "engineSize"]] = abs(df[["mpg", "engineSize"]]).round(1)
    try: 
        df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    except KeyError: # will raise keyError if we run it on testing data as it has no price column
        df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    
    return df

In [8]:
def decode(df, encoders):
    # Iterative imputer produces ~20-30 values that are outside of the range of the encoder
    # The simplest fix is to clip does values back into the range of the encoder
 

    df["Brand_transformed"] = df["Brand_transformed"].clip(lower=0, upper=encoders["Brand"].classes_.shape[0]-1).astype(int)
    df["Brand"] = encoders["Brand"].inverse_transform(df["Brand_transformed"])

    df["transmission_transformed"] = df["transmission_transformed"].clip(lower=0, upper=encoders["transmission"].classes_.shape[0]-1).astype(int)
    df["transmission"] = encoders["transmission"].inverse_transform(df["transmission_transformed"])
        
    # Use clip with the information of the fitted encoder, .classes_.shape gives us the dimension of the labels the encoder uses [0] is the rows - 1 because we start clipping at 0
    df["model_transformed"] = df["model_transformed"].clip(lower=0, upper=encoders["model"].classes_.shape[0]-1).astype(int)
    df["model"] = encoders["model"].inverse_transform(df["model_transformed"])
    
    df["fuelType_transformed"] = df["fuelType_transformed"].clip(lower=0, upper=encoders["fuelType"].classes_.shape[0]-1).astype(int)
    df["fuelType"] = encoders["fuelType"].inverse_transform(df["fuelType_transformed"])
    

    df.drop(columns=["Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"], inplace=True)

    return df

## Workflow for Train and Validation Sets

#### Encoding

In [40]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)
# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end

df_encoded, encoders = fit_transform_encoding(df)

#### Creating the stratification column

In [53]:
# Create Categorical price column with 0 < 1 < 2 for the price
df_encoded["price_cat"] = pd.qcut(df_encoded["price"], 3, labels=False)

# Combine the 10 unique brand values (1-9 & NA) with the 3 unique price category values (0-2)
stratify_col = df_encoded["Brand_transformed"].astype(str) + "_" + df_encoded["price_cat"].astype(str)

#### Imputation and decoding

In [54]:
# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42, stratify=stratify_col)
# train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42)

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputer_test = apply_imputer(validation_split, imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputer_test, encoders)

## Feature selection

In [12]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
y = df['price']

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Drop 'carID' and 'price'
num_cols = [col for col in numerical_cols 
            if "_transformed" not in col and col not in ['price', 'carID']]

print(num_cols)
print(categorical_cols)

['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners']
['Brand', 'model', 'transmission', 'fuelType']


In [13]:
#FILTER METHOD


#ANOVA FUNCTION

def anova_for_categorical(df, y, categorical_cols):
   
    f_scores, p_values = [], []

    for col in df.columns:
        if col in categorical_cols:
            groups = [y[df[col] == cat] for cat in df[col].dropna().unique()]
            if len(groups) > 1 and all(len(g) > 1 for g in groups):
                f_stat, p_val = stats.f_oneway(*groups)
            else:
                f_stat, p_val = 0.0, 1.0
        else:
            if df[col].nunique() > 1:
                corr = np.corrcoef(df[col], y)[0, 1]
                f_stat, p_val = corr**2 * len(y), 0.0
            else:
                f_stat, p_val = 0.0, 1.0
        f_scores.append(f_stat)
        p_values.append(p_val)

    return np.array(f_scores), np.array(p_values)

def filter_method_selection(X_train, y_train,
                            top_k=None,
                            var_threshold=0.01,
                            corr_threshold=0.85):

    
    
    print("FILTER METHOD (Variance + Spearman Correlation + ANOVA)")
    

    #categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    #num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # Variance Threshold
    vt_selector = VarianceThreshold(threshold=var_threshold)
    if num_cols:
        X_num_vt = pd.DataFrame(
            vt_selector.fit_transform(X_train[num_cols]),
            columns=np.array(num_cols)[vt_selector.get_support()]
        )
        print(f"Removed {len(num_cols) - X_num_vt.shape[1]} low-variance numeric features.")
    else:
        X_num_vt = pd.DataFrame(index=df.index)

    # Spearman Correlation 
    if not X_num_vt.empty:
        corr_matrix = X_num_vt.corr(method='spearman').abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
        X_num_corr = X_num_vt.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} correlated numeric features (Spearman |corr| > {corr_threshold}).")
    else:
        X_num_corr = X_num_vt

    # Combine numeric + categorical
    X_filtered = pd.concat([X_num_corr, df[categorical_cols]], axis=1)

    # ANOVA F-test
    f_scores, f_pvalues = anova_for_categorical(X_filtered, y, categorical_cols)
    f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-10)

    results_df = pd.DataFrame({
        'feature': X_filtered.columns,
        'ANOVA_F': f_scores,
        'ANOVA_p_value': f_pvalues,
        'ANOVA_norm': f_norm,
        'type': ['categorical' if c in categorical_cols else 'numerical' for c in X_filtered.columns]
    }).sort_values('ANOVA_norm', ascending=False)

    if top_k is None:
        selected_features = X_filtered.columns.tolist()
    else:
        selected_features = X_filtered.columns[np.argsort(f_norm)[-top_k:]].tolist()

    print(f"\n Filter method selected {len(selected_features)} features")
    return selected_features, X_filtered[selected_features], results_df




In [14]:
#WRAPPER METHOD

def rfe(train_processed, validation_processed, num_cols, step=1, n_estimators=100, random_state=42):
    
    # Filter numeric columns to those that actually exist in train_processed
    valid_num_cols = [col for col in num_cols if col in train_processed.columns]
    if len(valid_num_cols) == 0:
        raise ValueError("No valid numeric columns found in train_processed.")

    print(f"Using {len(valid_num_cols)} numeric columns for RFE:\n{valid_num_cols}")

    # Prepare numeric features and targets
    X_train_num = train_processed[valid_num_cols]
    y_train = train_processed['price']

    X_val_num = validation_processed[valid_num_cols]
    y_val = validation_processed['price']

    nof_list = np.arange(1, X_train_num.shape[1]+1)
    high_score = 0
    nof = 0
    train_score_list = []
    val_score_list = []
    features_to_select = None

    for n in nof_list:
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
        rfe = RFE(estimator=model, n_features_to_select=n, step=step)
        X_train_rfe = rfe.fit_transform(X_train_num, y_train)
        X_val_rfe = rfe.transform(X_val_num)
        model.fit(X_train_rfe, y_train)

        train_score = model.score(X_train_rfe, y_train)
        val_score = model.score(X_val_rfe, y_val)
        train_score_list.append(train_score)
        val_score_list.append(val_score)

        if val_score > high_score:
            high_score = val_score
            nof = n
            features_to_select = pd.Series(rfe.support_, index=X_train_num.columns)

    selected_features = features_to_select[features_to_select].index.tolist()
    
    print
    print(f"\nOptimum number of features: {nof}")
    print(f"Best validation score: {high_score:.4f}")
    print("Selected features:")
    print(selected_features)

    return nof, high_score, selected_features, train_score_list, val_score_list


In [15]:

X_train = df[num_cols]
y_train = df['price']

# Filter method (Variance + Spearman + ANOVA)
selected_filter, X_train_filtered, filter_df = filter_method_selection(
    X_train, y_train,
    top_k=None, var_threshold=0.01, corr_threshold=0.85
)

#Print features selected

print("\nFeatures selected by Filter Method:")
for f in selected_filter:
    print(f)

# RFE (Random Forest) using your workflow variables
nof, best_score, selected_rfe_features, train_scores, val_scores = rfe(
    train_processed, validation_processed, num_cols
)


#print("\nFeatures ranked by RFE (best first):")
#for f in selected_rfe_features:
   #print(f)


FILTER METHOD (Variance + Spearman Correlation + ANOVA)
Removed 0 low-variance numeric features.
Removed 0 correlated numeric features (Spearman |corr| > 0.85).

 Filter method selected 11 features

Features selected by Filter Method:
year
mileage
tax
mpg
engineSize
paintQuality%
previousOwners
Brand
model
transmission
fuelType
Using 7 numeric columns for RFE:
['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners']

Optimum number of features: 7
Best validation score: 0.8786
Selected features:
['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners']


## Workflow for Seperated Testing Dataset

In [16]:
test_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/test.csv")

In [17]:
test_df.set_index("carID")

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
105775,VW,Tiguan,2017.000000,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
81363,BMW,X2,2020.000000,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
76833,Audi,Q5,2019.000000,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0


In [18]:
test = simple_processing(test_df)
test_df_encoded, test_encoders = fit_transform_encoding(test)


df_encoded_no_price = df_encoded.drop(["price"], axis=1)
test_imputers = fit_imputer(df_encoded_no_price) # we use the full encoded training dataframe to train the imputers

# Apply trained imputer to both datasplits
test_df_imputed = apply_imputer(test_df_encoded, test_imputers)

test_processed = decode(test_df_imputed, test_encoders)


In [19]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

c:\Users\morit\anaconda3\envs\DataMining\Lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Test of the models on the full dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

class BrandModelTrainer:
    def __init__(self, estimator):
        self.estimator = estimator
        self.brand_models = {}
        self.feature_cols = None

    def fit(self, X_train, y_train):
        self.feature_cols = [c for c in X_train.columns if c != "Brand"]
        print(f"Training models for {len(X_train['Brand'].unique())} brands...\n")

        for brand in X_train["Brand"].unique():
            mask = X_train["Brand"] == brand
            Xb = X_train.loc[mask, self.feature_cols]
            yb = y_train[mask]

            numeric_cols = Xb.select_dtypes(include=["int64", "float64"]).columns.tolist()
            categorical_cols = Xb.select_dtypes(include=["object", "category"]).columns.tolist()

            preprocessor = ColumnTransformer([
                ("num", RobustScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
            ])

            model = Pipeline([
                ("preprocess", preprocessor),
                ("estimator", clone(self.estimator))
            ])

            model.fit(Xb, yb)
            self.brand_models[brand] = model
            print(f"  ✓ {brand} done.")

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for brand, model in self.brand_models.items():
            mask = X["Brand"] == brand
            if mask.sum() == 0:
                continue
            Xb = X.loc[mask, self.feature_cols]
            preds[mask] = model.predict(Xb)
        return preds

    # --- metriche overall ---
    def evaluate_train(self, X_train, y_train):
        y_pred = self.predict(X_train)
        rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        mae = mean_absolute_error(y_train, y_pred)
        r2 = r2_score(y_train, y_pred)
        print("\nTraining Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate(self, X_val, y_val):
        y_pred = self.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        print("\nValidation Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    # --- metriche per brand ---
    def evaluate_by_brand(self, X, y, split_name="Validation"):
        y_pred = self.predict(X)
        results = []
        for brand in X["Brand"].unique():
            mask = X["Brand"] == brand
            y_true_b = y[mask]
            y_pred_b = y_pred[mask]
            rmse = np.sqrt(mean_squared_error(y_true_b, y_pred_b))
            mae = mean_absolute_error(y_true_b, y_pred_b)
            r2 = r2_score(y_true_b, y_pred_b)
            results.append({"Brand": brand, "N": len(y_true_b), "RMSE": rmse, "MAE": mae, "R²": r2})
        df = pd.DataFrame(results).sort_values("RMSE")
        print(f"\n{split_name} Performance per Brand:")
        print(df.to_string(index=False))
        return df

    def evaluate_train_by_brand(self, X_train, y_train):
        return self.evaluate_by_brand(X_train, y_train, split_name="Training")


In [ ]:
X_train = train_processed.drop(columns=['price'])
y_train = train_processed['price']
X_val = validation_processed.drop(columns=['price'])
y_val = validation_processed['price']

### Linear Regression

In [ ]:
# istanza del modello
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ ford done.
  ✓ opel done.
  ✓ toyota done.
  ✓ audi done.
  ✓ vw done.
  ✓ bmw done.
  ✓ skoda done.
  ✓ mercedes done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 3573.35
  MAE:  2147.99
  R²:   0.8666

Validation Set Performance (Overall):
  RMSE: 3603.71
  MAE:  2200.42
  R²:   0.8574

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7632 1533.549775 1081.423830 0.813446
  toyota  3779 1974.626957 1201.397292 0.904956
    ford 13210 2170.103157 1538.185280 0.794413
 hyundai  2688 2229.824363 1584.147038 0.860903
   skoda  3472 2420.399409 1531.018872 0.853623
      vw  8489 2967.956568 2098.474300 0.855281
     bmw  6026 4498.733500 2998.509959 0.842406
    audi  5941 4516.994825 2898.436554 0.854849
mercedes  9541 5876.778979 3643.353534 0.728465

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1780.798455 1163.555551 0.757113
  toyota  939

,Brand,N,RMSE,MAE,R²
6,opel,1912,1780.798455,1163.555551,0.757113
8,toyota,939,1953.615747,1185.807427,0.901940
3,ford,3182,2148.148915,1571.776300,0.798188
1,skoda,914,2215.800666,1620.722785,0.859080
7,hyundai,716,2324.482949,1627.651243,0.842722
2,vw,2117,3011.647157,2090.963644,0.836544
4,audi,1527,4295.372926,2858.500145,0.844805
5,bmw,1520,4565.116790,3186.136409,0.852051
0,mercedes,2368,6012.190900,3722.365153,0.691775


### SGD


In [ ]:
# istanza del modello
SGD = SGDRegressor()
trainer = BrandModelTrainer(SGD)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID


Training models for 9 brands...

  ✓ ford done.
  ✓ opel done.
  ✓ toyota done.
  ✓ audi done.
  ✓ vw done.
  ✓ bmw done.
  ✓ skoda done.
  ✓ mercedes done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 3700.61
  MAE:  2202.12
  R²:   0.8570

Validation Set Performance (Overall):
  RMSE: 3673.38
  MAE:  2238.43
  R²:   0.8518

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7632 1566.175417 1097.382237 0.805424
  toyota  3779 2105.819882 1288.075687 0.891907
    ford 13210 2188.660130 1541.048406 0.790882
 hyundai  2688 2267.626697 1613.936116 0.856146
   skoda  3472 2462.407195 1561.619080 0.848498
      vw  8489 3030.022788 2126.741936 0.849165
    audi  5941 4737.758998 2983.818761 0.840314
     bmw  6026 4778.342270 3185.031787 0.822208
mercedes  9541 6055.090294 3721.421740 0.711738

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1786.683570 1170.552144 0.755505
  toyota  939

,Brand,N,RMSE,MAE,R²
6,opel,1912,1786.683570,1170.552144,0.755505
8,toyota,939,2066.496833,1255.066828,0.890280
3,ford,3182,2145.149893,1568.618647,0.798751
1,skoda,914,2203.476935,1611.384501,0.860643
7,hyundai,716,2330.153340,1640.869594,0.841954
2,vw,2117,3041.030734,2100.352538,0.833339
4,audi,1527,4419.672184,2945.906111,0.835693
5,bmw,1520,4699.762206,3308.282664,0.843195
0,mercedes,2368,6129.806976,3793.843113,0.679598


### KNR

In [ ]:
# istanza del modello
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ ford done.
  ✓ opel done.
  ✓ toyota done.
  ✓ audi done.
  ✓ vw done.
  ✓ bmw done.
  ✓ skoda done.
  ✓ mercedes done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 2470.81
  MAE:  1429.16
  R²:   0.9362

Validation Set Performance (Overall):
  RMSE: 2871.05
  MAE:  1754.40
  R²:   0.9095

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7632 1115.324646  767.454271 0.901324
    ford 13210 1391.385282  925.973460 0.915486
  toyota  3779 1394.726467  850.620323 0.952583
 hyundai  2688 1566.784963 1008.056845 0.931325
   skoda  3472 1928.556702 1168.078111 0.907068
      vw  8489 2077.856596 1421.498245 0.929068
    audi  5941 3239.159895 2063.018717 0.925357
     bmw  6026 3550.559728 2170.960903 0.901836
mercedes  9541 3749.891679 2241.560214 0.889444

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1532.825684 1020.962134 0.820046
  toyota  939

,Brand,N,RMSE,MAE,R²
6,opel,1912,1532.825684,1020.962134,0.820046
8,toyota,939,1626.004099,1051.836209,0.932070
3,ford,3182,1665.796272,1145.301760,0.878644
7,hyundai,716,1788.807847,1227.449162,0.906859
1,skoda,914,2086.463975,1481.335449,0.875051
2,vw,2117,2491.049855,1737.508455,0.888171
4,audi,1527,3724.407704,2443.301506,0.883321
0,mercedes,2368,4115.034353,2640.656672,0.855606
5,bmw,1520,4349.842562,2749.307500,0.865676


### Random Forest

In [ ]:
# istanza del modello
RF = RandomForestRegressor()
trainer = BrandModelTrainer(RF)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ ford done.
  ✓ opel done.
  ✓ toyota done.
  ✓ audi done.
  ✓ vw done.
  ✓ bmw done.
  ✓ skoda done.
  ✓ mercedes done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 851.22
  MAE:  485.15
  R²:   0.9924

Validation Set Performance (Overall):
  RMSE: 2161.39
  MAE:  1299.76
  R²:   0.9487

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7632  437.038738 291.279999 0.984849
    ford 13210  536.245360 345.802193 0.987447
 hyundai  2688  585.462600 366.681894 0.990411
  toyota  3779  601.721404 350.428799 0.991174
      vw  8489  748.160593 465.990654 0.990804
   skoda  3472  768.581489 420.593376 0.985240
    audi  5941 1060.415980 676.493977 0.992000
     bmw  6026 1188.625681 669.998342 0.988999
mercedes  9541 1232.589609 724.535018 0.988055

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1264.632785  852.673483 0.877509
    ford 3182 1348.644188

,Brand,N,RMSE,MAE,R²
6,opel,1912,1264.632785,852.673483,0.877509
3,ford,3182,1348.644188,936.011037,0.920455
7,hyundai,716,1441.565837,988.105140,0.939510
8,toyota,939,1537.096230,925.097220,0.939296
1,skoda,914,1685.156689,1132.800733,0.918494
2,vw,2117,1835.511949,1219.519901,0.939284
4,audi,1527,2815.877301,1692.914283,0.933303
5,bmw,1520,2947.772297,1870.819592,0.938313
0,mercedes,2368,3142.848884,1908.433226,0.915773


### Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=1000,
    batch_size=32,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,  # Più pazienza
    validation_fraction=0.15,
    verbose=True
)

trainer = BrandModelTrainer(mlp_deep)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 17265756.71320347
Validation score: 0.822128
Iteration 2, loss = 2008471.22345463
Validation score: 0.845592
Iteration 3, loss = 1823006.24865212
Validation score: 0.849938
Iteration 4, loss = 1751040.61808909
Validation score: 0.861746
Iteration 5, loss = 1682686.63626834
Validation score: 0.864496
Iteration 6, loss = 1651009.64549285
Validation score: 0.864876
Iteration 7, loss = 1601359.42681784
Validation score: 0.874147
Iteration 8, loss = 1598321.72546553
Validation score: 0.875085
Iteration 9, loss = 1553618.65811316
Validation score: 0.846837
Iteration 10, loss = 1537157.23055374
Validation score: 0.881797
Iteration 11, loss = 1516138.98440866
Validation score: 0.879262
Iteration 12, loss = 1503630.27743334
Validation score: 0.876335
Iteration 13, loss = 1492015.24555670
Validation score: 0.882752
Iteration 14, loss = 1448849.76032266
Validation score: 0.880732
Iteration 15, loss = 1442120.53374563
Validation score: 0.883179


c:\Users\morit\anaconda3\envs\DataMining\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


  ✓ opel done.
Iteration 1, loss = 61144061.43157237
Validation score: 0.667660
Iteration 2, loss = 4087910.57360494
Validation score: 0.847113
Iteration 3, loss = 2850769.39289294
Validation score: 0.882927
Iteration 4, loss = 2395638.43716217
Validation score: 0.884702
Iteration 5, loss = 2166144.45344584
Validation score: 0.909215
Iteration 6, loss = 1939578.04393248
Validation score: 0.917867
Iteration 7, loss = 1793979.79661260
Validation score: 0.921523
Iteration 8, loss = 1654805.63615271
Validation score: 0.926142
Iteration 9, loss = 1586568.29990813
Validation score: 0.929131
Iteration 10, loss = 1503950.81967401
Validation score: 0.932146
Iteration 11, loss = 1449686.27181900
Validation score: 0.932549
Iteration 12, loss = 1376967.49321040
Validation score: 0.937386
Iteration 13, loss = 1356679.98648203
Validation score: 0.937405
Iteration 14, loss = 1329602.98211842
Validation score: 0.938327
Iteration 15, loss = 1294143.31127166
Validation score: 0.938867
Iteration 16, loss

: 

: 

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_wide = MLPRegressor(
    hidden_layer_sizes=(1024,),  # Un solo layer MOLTO largo
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=1000,
    batch_size=64,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,
    validation_fraction=0.15,
    verbose=True
)

trainer = BrandModelTrainer(mlp_wide)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)